# 2D Gaussian Splatting — Mitsubishi Logo Pipeline

Blender synthetic 미쓰비시 로고 데이터를 2DGS로 학습/렌더링/메쉬 추출하는 전체 파이프라인.

**데이터**: 641뷰, 800x800, RGB + GT Depth + Calibration (calib ini)

**핵심 변환**: calib ini → COLMAP binary format + GT depth → initial point cloud

## 1. Environment Setup

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Google Drive 마운트 (데이터/체크포인트 영속 저장용)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE = "/content/drive/MyDrive/2dgs_results"
import os
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f"Drive mounted. Save path: {DRIVE_SAVE}")

In [ ]:
import os
ROOT = "/content"
REPO = os.path.join(ROOT, "2d-gaussian-splatting")
DATA = os.path.join(ROOT, "mitsubishi")  # 최종 COLMAP 포맷 데이터 경로
OUTPUT = os.path.join(ROOT, "output")     # 학습 출력 경로

In [ ]:
# 레포 클론 (본인 fork — 스케일 정규화 파이프라인 포함)
if not os.path.exists(REPO):
    !git clone https://github.com/BAEJUNHAK/2d-gaussian-splatting.git --recursive {REPO}

In [ ]:
# PyTorch 버전 확인 (Colab 기본 설치된 것 사용)
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# 의존성 설치 (environment.yml 기반)
!pip install -q plyfile scikit-image==0.21.0 lpips==0.1.4 trimesh==4.3.2 \
    open3d mediapy==1.1.2 opencv-python tqdm pillow tensorboard

# simple_knn 빌드 수정: CUDA 12.x에서 FLT_MAX 정의 누락 문제
import subprocess
knn_cu = os.path.join(REPO, "submodules", "simple-knn", "simple_knn.cu")
with open(knn_cu, 'r') as f:
    src = f.read()
if '#include <cfloat>' not in src:
    src = src.replace('#include "simple_knn.h"', '#include "simple_knn.h"\n#include <cfloat>')
    with open(knn_cu, 'w') as f:
        f.write(src)
    print("Patched simple_knn.cu: added #include <cfloat>")

# CUDA 서브모듈 빌드
!pip install -q {REPO}/submodules/diff-surfel-rasterization
!pip install -q {REPO}/submodules/simple-knn

## 2. Data Download & Extraction

In [ ]:
!pip install -q gdown
import gdown

RAW = os.path.join(ROOT, "raw_data")
os.makedirs(RAW, exist_ok=True)

# RGB + Camera
rgb_zip = os.path.join(RAW, "dataset.zip")
if not os.path.exists(rgb_zip):
    gdown.download(id="1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB", output=rgb_zip, quiet=False)

# Depth
depth_zip = os.path.join(RAW, "depth_data.zip")
if not os.path.exists(depth_zip):
    gdown.download(id="1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ", output=depth_zip, quiet=False)

In [ ]:
import zipfile

# 압축 해제
with zipfile.ZipFile(rgb_zip, 'r') as z:
    z.extractall(RAW)
with zipfile.ZipFile(depth_zip, 'r') as z:
    z.extractall(RAW)

RGB_DIR = os.path.join(RAW, "dataset")
DEPTH_DIR = os.path.join(RAW, "dataset_depth")

# 파일 수 확인
rgb_files = sorted([f for f in os.listdir(RGB_DIR) if f.startswith("rgb_")])
calib_files = sorted([f for f in os.listdir(RGB_DIR) if f.startswith("calib_")])
depth_files = sorted([f for f in os.listdir(DEPTH_DIR) if f.startswith("depth_")])
print(f"RGB: {len(rgb_files)}, Calib: {len(calib_files)}, Depth: {len(depth_files)}")

## 3. Data Conversion — calib ini → COLMAP binary

**스케일 정규화가 필수!** CUDA 래스터라이저에 `FAR_PLANE=100.0`이 하드코딩.
원본 씬 depth(210~295m)는 이 범위를 초과해서 depth mapping이 깨짐.
→ 전체 좌표를 ÷100으로 정규화 (depth 2.1~2.95m, 카메라 반경 4.0)

2DGS가 기대하는 디렉토리 구조:
```
mitsubishi/
├── images/          ← RGB 이미지 (배경=검정)
├── sparse/0/        ← COLMAP reconstruction (정규화된 좌표)
│   ├── cameras.bin
│   ├── images.bin
│   └── points3D.bin
└── depth_gt/        ← GT depth (평가용)
```

In [ ]:
import numpy as np
import struct
import shutil
from PIL import Image
from configparser import ConfigParser


def parse_calib_ini(filepath):
    """
    calib_XXXX.ini 파일에서 K, R, t를 추출.
    Returns: K (3x3), R (3x3), t (3,), width, height
    """
    # 주석 행 제거 후 파싱
    with open(filepath, 'r') as f:
        lines = [l for l in f.readlines() if not l.startswith('#')]
    content = '\n'.join(lines)

    cp = ConfigParser()
    cp.read_string(content)
    sec = 'camera_0'

    # k_matrix=Matrix:3:3:v0,v1,...,v8
    k_str = cp.get(sec, 'k_matrix')
    k_vals = list(map(float, k_str.split('Matrix:3:3:')[1].split(',')))
    K = np.array(k_vals).reshape(3, 3)

    # r_matrix=Matrix:3:3:v0,...,v8
    r_str = cp.get(sec, 'r_matrix')
    r_vals = list(map(float, r_str.split('Matrix:3:3:')[1].split(',')))
    R = np.array(r_vals).reshape(3, 3)

    # t_vector=Vector:3:v0,v1,v2
    t_str = cp.get(sec, 't_vector')
    t_vals = list(map(float, t_str.split('Vector:3:')[1].split(',')))
    t = np.array(t_vals)

    w = int(cp.get(sec, 'width'))
    h = int(cp.get(sec, 'height'))

    return K, R, t, w, h


def rotmat2qvec(R):
    """3x3 rotation matrix → quaternion (w, x, y, z). COLMAP convention."""
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec


# 테스트: 첫 번째 calib 파일 파싱
K0, R0, t0, w0, h0 = parse_calib_ini(os.path.join(RGB_DIR, calib_files[0]))
print(f"K:\n{K0}")
print(f"\nR:\n{R0}")
print(f"\nt: {t0}")
print(f"\nImage size: {w0}x{h0}")
print(f"\nfx={K0[0,0]:.3f}, fy={K0[1,1]:.3f}, cx={K0[0,2]:.1f}, cy={K0[1,2]:.1f}")

In [ ]:
###############################################
# COLMAP binary writers
###############################################

def write_cameras_binary(cam_dict, path):
    """
    cam_dict: {camera_id: {'model_id': int, 'width': int, 'height': int, 'params': [float]}}
    PINHOLE model_id = 1, params = [fx, fy, cx, cy]
    """
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(cam_dict)))
        for cam_id, cam in cam_dict.items():
            f.write(struct.pack('<iiQQ',
                               cam_id, cam['model_id'], cam['width'], cam['height']))
            for p in cam['params']:
                f.write(struct.pack('<d', p))


def write_images_binary(img_dict, path):
    """
    img_dict: {image_id: {'qvec': (4,), 'tvec': (3,), 'camera_id': int, 'name': str}}
    """
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(img_dict)))
        for img_id, img in img_dict.items():
            # image_id(i) + qvec(4d) + tvec(3d) + camera_id(i) = 64 bytes
            f.write(struct.pack('<i', img_id))
            for q in img['qvec']:
                f.write(struct.pack('<d', q))
            for t in img['tvec']:
                f.write(struct.pack('<d', t))
            f.write(struct.pack('<i', img['camera_id']))
            # name: null-terminated string
            f.write(img['name'].encode('utf-8'))
            f.write(b'\x00')
            # num_points2D = 0 (no 2D-3D correspondences)
            f.write(struct.pack('<Q', 0))


def write_points3D_binary(pts_dict, path):
    """
    pts_dict: {point3D_id: {'xyz': (3,), 'rgb': (3,), 'error': float}}
    """
    with open(path, 'wb') as f:
        f.write(struct.pack('<Q', len(pts_dict)))
        for pt_id, pt in pts_dict.items():
            f.write(struct.pack('<Q', pt_id))
            for x in pt['xyz']:
                f.write(struct.pack('<d', x))
            for c in pt['rgb']:
                f.write(struct.pack('<B', int(c)))
            f.write(struct.pack('<d', pt['error']))
            # track_length = 0
            f.write(struct.pack('<Q', 0))


print("COLMAP binary writers defined.")

In [ ]:
###############################################
# 디렉토리 구조 생성 + RGB 이미지 (원본) + COLMAP 파일 (스케일 정규화)
###############################################

# 스케일 정규화 상수
SCALE_FACTOR = 100.0
CENTER = np.array([0.0, 0.0, 0.0])

# 디렉토리 생성
os.makedirs(os.path.join(DATA, "images"), exist_ok=True)
os.makedirs(os.path.join(DATA, "sparse", "0"), exist_ok=True)
os.makedirs(os.path.join(DATA, "depth_gt"), exist_ok=True)

# --- 1) cameras.bin ---
fx, fy = K0[0, 0], K0[1, 1]
cx, cy = K0[0, 2], K0[1, 2]

cameras = {
    1: {
        'model_id': 1,   # PINHOLE
        'width': w0,
        'height': h0,
        'params': [fx, fy, cx, cy]
    }
}
write_cameras_binary(cameras, os.path.join(DATA, "sparse", "0", "cameras.bin"))
print(f"cameras.bin written: PINHOLE {w0}x{h0}, fx={fx:.3f}, fy={fy:.3f}, cx={cx:.1f}, cy={cy:.1f}")

# --- 2) images.bin (정규화된 t) + RGB 이미지 (원본 그대로) ---
# 배경 제거하지 않음! 이유:
# - 전경 11.2% + 단색 → 배경=검정으로 하면 opacity gradient 불균형으로 전체 pruning
# - 원본 회색 배경(~70/255) 유지 → 모든 픽셀에서 유의미한 GT → 안정적 학습
images = {}
for i, (rgb_f, cal_f) in enumerate(zip(rgb_files, calib_files)):
    dst = os.path.join(DATA, "images", rgb_f)
    if not os.path.exists(dst):
        shutil.copy2(os.path.join(RGB_DIR, rgb_f), dst)

    # calib 파싱
    K, R, t, _, _ = parse_calib_ini(os.path.join(RGB_DIR, cal_f))
    qvec = rotmat2qvec(R)
    t_normalized = (R @ CENTER + t) / SCALE_FACTOR

    images[i + 1] = {
        'qvec': qvec,
        'tvec': t_normalized,
        'camera_id': 1,
        'name': rgb_f
    }

write_images_binary(images, os.path.join(DATA, "sparse", "0", "images.bin"))
print(f"images.bin written: {len(images)} images (t ÷ {SCALE_FACTOR})")

# --- 3) Depth 이미지 복사 (평가용) ---
for df in depth_files:
    src = os.path.join(DEPTH_DIR, df)
    dst = os.path.join(DATA, "depth_gt", df)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
print(f"GT depth copied: {len(depth_files)} files")

In [ ]:
###############################################
# GT Depth로부터 초기 포인트 클라우드 생성 (정규화 좌표)
###############################################
from tqdm import tqdm

# 매 10번째 뷰에서 depth back-projection
SAMPLE_EVERY_N_VIEWS = 10
PTS_PER_VIEW = 2000

all_points = []
all_colors = []

sampled_indices = list(range(0, len(rgb_files), SAMPLE_EVERY_N_VIEWS))
print(f"Sampling from {len(sampled_indices)} views...")
print(f"Scale factor: {SCALE_FACTOR}, Center: {CENTER}")

for idx in tqdm(sampled_indices):
    # depth 로드 (uint16, raw * 0.01 = meters)
    depth_path = os.path.join(DATA, "depth_gt", depth_files[idx])
    depth_raw = np.array(Image.open(depth_path)).astype(np.float64)
    # 정규화된 depth: (raw * 0.01) / SCALE_FACTOR
    depth_norm = depth_raw * 0.01 / SCALE_FACTOR

    # RGB 로드
    rgb_path = os.path.join(DATA, "images", rgb_files[idx])
    rgb = np.array(Image.open(rgb_path))

    # 전경 마스크
    fg_mask = depth_raw > 0
    fg_v, fg_u = np.where(fg_mask)

    if len(fg_v) == 0:
        continue

    # 랜덤 서브샘플링
    n_sample = min(PTS_PER_VIEW, len(fg_v))
    chosen = np.random.choice(len(fg_v), n_sample, replace=False)
    u = fg_u[chosen].astype(np.float64)
    v = fg_v[chosen].astype(np.float64)
    z = depth_norm[fg_v[chosen], fg_u[chosen]]  # 정규화된 depth

    # 카메라 좌표로 back-projection (정규화된 z 사용)
    x_cam = (u - cx) * z / fx
    y_cam = (v - cy) * z / fy
    z_cam = z
    pts_cam = np.stack([x_cam, y_cam, z_cam], axis=1)

    # 월드 좌표로 변환 (정규화된 t 사용)
    cal_path = os.path.join(RGB_DIR, calib_files[idx])
    _, R, t, _, _ = parse_calib_ini(cal_path)
    t_norm = (R @ CENTER + t) / SCALE_FACTOR
    pts_world = (R.T @ (pts_cam - t_norm).T).T

    # 색상
    colors = rgb[fg_v[chosen], fg_u[chosen], :3]

    all_points.append(pts_world)
    all_colors.append(colors)

all_points = np.concatenate(all_points, axis=0)
all_colors = np.concatenate(all_colors, axis=0)
print(f"Total raw points: {len(all_points)}")

# Voxel downsampling (정규화된 스케일에 맞춤)
VOXEL_SIZE = 0.01  # 오브젝트 크기 ~3x2.7x1.05 (정규화 후)
voxel_coords = np.floor(all_points / VOXEL_SIZE).astype(np.int64)
_, unique_idx = np.unique(voxel_coords, axis=0, return_index=True)
all_points = all_points[unique_idx]
all_colors = all_colors[unique_idx]
print(f"After voxel downsampling (voxel={VOXEL_SIZE}): {len(all_points)} points")
print(f"Point cloud range: [{all_points.min(0)}] ~ [{all_points.max(0)}]")

# points3D.bin 쓰기
pts_dict = {}
for i in range(len(all_points)):
    pts_dict[i + 1] = {
        'xyz': all_points[i],
        'rgb': all_colors[i],
        'error': 0.0
    }

write_points3D_binary(pts_dict, os.path.join(DATA, "sparse", "0", "points3D.bin"))
print(f"points3D.bin written: {len(pts_dict)} points")

In [ ]:
# 변환 결과 검증
import sys
sys.path.insert(0, REPO)

from scene.colmap_loader import (
    read_intrinsics_binary, read_extrinsics_binary, read_points3D_binary
)

sparse_dir = os.path.join(DATA, "sparse", "0")

# cameras.bin 검증
cams = read_intrinsics_binary(os.path.join(sparse_dir, "cameras.bin"))
for cid, c in cams.items():
    print(f"Camera {cid}: {c.model} {c.width}x{c.height} params={c.params}")

# images.bin 검증
imgs = read_extrinsics_binary(os.path.join(sparse_dir, "images.bin"))
print(f"\nImages loaded: {len(imgs)}")
sample_img = imgs[1]
print(f"  First: id={sample_img.id}, name={sample_img.name}")
print(f"  qvec={sample_img.qvec}")
print(f"  tvec={sample_img.tvec}")

# points3D.bin 검증
xyz, rgb, err = read_points3D_binary(os.path.join(sparse_dir, "points3D.bin"))
print(f"\nPoint cloud: {xyz.shape[0]} points")
print(f"  XYZ range: [{xyz.min(0)}, {xyz.max(0)}]")
print(f"  RGB range: [{rgb.min()}, {rgb.max()}]")

In [ ]:
# 포인트 클라우드 시각화 (3D scatter)
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 5))

# 포인트 클라우드
ax1 = fig.add_subplot(121, projection='3d')
subsample = np.random.choice(len(xyz), min(5000, len(xyz)), replace=False)
ax1.scatter(xyz[subsample, 0], xyz[subsample, 1], xyz[subsample, 2],
            c=rgb[subsample] / 255.0, s=0.5)
ax1.set_title(f'Point Cloud ({len(xyz)} pts)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# 카메라 위치
ax2 = fig.add_subplot(122, projection='3d')
cam_positions = []
for img_id, img in imgs.items():
    from scene.colmap_loader import qvec2rotmat
    R = qvec2rotmat(img.qvec)
    cam_pos = -R.T @ img.tvec
    cam_positions.append(cam_pos)
cam_positions = np.array(cam_positions)
ax2.scatter(cam_positions[:, 0], cam_positions[:, 1], cam_positions[:, 2], s=1, alpha=0.5)
ax2.scatter(xyz[subsample, 0], xyz[subsample, 1], xyz[subsample, 2],
            c='gray', s=0.2, alpha=0.3)
ax2.set_title(f'Cameras ({len(cam_positions)}) + Points')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

plt.tight_layout()
plt.show()

## 4. Training

**배경 처리:**
- 원본 이미지 그대로 사용 (회색 배경 ~70/255 유지)
- 배경 제거(→검정)하면 전경 11.2% + 단색이라 opacity 붕괴 → 전체 pruning
- 원본 배경 유지 시 모든 픽셀에서 유의미한 loss → 안정적 학습

**하이퍼파라미터:**
- `lambda_normal=0.05`: 텍스처 없는 씬에서 geometry 안정화 필수
- `lambda_dist=0.1`: depth distortion 정규화
- `depth_ratio=1`: bounded scene → median depth 사용
- `sh_degree=1`: 거의 단색 오브젝트이므로 고차 SH 불필요

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT}

In [ ]:
!cd {REPO} && python train.py \
    -s {DATA} \
    -m {OUTPUT} \
    --eval \
    --sh_degree 1 \
    --lambda_normal 0.05 \
    --lambda_dist 0.1 \
    --depth_ratio 1.0 \
    --iterations 30000 \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 30000 \
    --checkpoint_iterations 15000 30000

In [ ]:
# 체크포인트를 Google Drive에 백업
!cp -r {OUTPUT}/point_cloud {DRIVE_SAVE}/ 2>/dev/null
!cp -r {OUTPUT}/cfg_args {DRIVE_SAVE}/ 2>/dev/null
!cp -r {OUTPUT}/chkpnt*.pth {DRIVE_SAVE}/ 2>/dev/null
print(f"Checkpoint saved to {DRIVE_SAVE}")

## 5. Rendering & Evaluation

In [ ]:
# 테스트 뷰 렌더링 (메쉬 추출은 다음 섹션)
!cd {REPO} && python render.py \
    -m {OUTPUT} \
    -s {DATA} \
    --eval \
    --depth_ratio 1.0 \
    --skip_mesh

In [ ]:
# 정량 평가 (PSNR, SSIM, LPIPS)
!cd {REPO} && python metrics.py -m {OUTPUT}

In [ ]:
# 렌더링 결과 시각화: GT vs Rendered
import glob
import matplotlib.pyplot as plt

# 마지막 iteration 찾기
test_dirs = sorted(glob.glob(os.path.join(OUTPUT, "test", "ours_*")))
if test_dirs:
    test_dir = test_dirs[-1]
    render_dir = os.path.join(test_dir, "renders")
    gt_dir = os.path.join(test_dir, "gt")

    renders = sorted(glob.glob(os.path.join(render_dir, "*.png")))[:8]
    gts = sorted(glob.glob(os.path.join(gt_dir, "*.png")))[:8]

    fig, axes = plt.subplots(2, min(8, len(renders)), figsize=(20, 5))
    if len(renders) == 1:
        axes = axes.reshape(2, 1)
    for i, (r, g) in enumerate(zip(renders, gts)):
        axes[0, i].imshow(np.array(Image.open(g)))
        axes[0, i].set_title('GT', fontsize=8)
        axes[0, i].axis('off')
        axes[1, i].imshow(np.array(Image.open(r)))
        axes[1, i].set_title('Rendered', fontsize=8)
        axes[1, i].axis('off')
    plt.suptitle(f'Test Views ({os.path.basename(test_dir)})')
    plt.tight_layout()
    plt.show()
else:
    print("No test results found.")

In [ ]:
# Depth/Normal 시각화
if test_dirs:
    vis_dir = os.path.join(test_dirs[-1], "vis")
    depth_vis = sorted(glob.glob(os.path.join(vis_dir, "depth_*.tiff")))[:4]

    if depth_vis:
        fig, axes = plt.subplots(1, len(depth_vis), figsize=(16, 4))
        if len(depth_vis) == 1:
            axes = [axes]
        for i, dp in enumerate(depth_vis):
            d = np.array(Image.open(dp))
            axes[i].imshow(d, cmap='turbo')
            axes[i].set_title(f'Depth {i}', fontsize=8)
            axes[i].axis('off')
        plt.suptitle('Rendered Depth Maps')
        plt.tight_layout()
        plt.show()

## 6. Mesh Extraction

Bounded mesh extraction (depth_ratio=1로 학습했으므로).

In [ ]:
# Bounded mesh extraction
!cd {REPO} && python render.py \
    -m {OUTPUT} \
    -s {DATA} \
    --depth_ratio 1.0 \
    --skip_train \
    --skip_test \
    --num_cluster 50

In [ ]:
# 메쉬 시각화
import trimesh

train_dirs = sorted(glob.glob(os.path.join(OUTPUT, "train", "ours_*")))
if train_dirs:
    mesh_path = os.path.join(train_dirs[-1], "fuse_post.ply")
    if not os.path.exists(mesh_path):
        mesh_path = os.path.join(train_dirs[-1], "fuse.ply")

    if os.path.exists(mesh_path):
        mesh = trimesh.load(mesh_path)
        print(f"Mesh loaded: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")
        print(f"Bounding box: {mesh.bounds}")

        # 여러 각도에서 렌더링
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for i, (elev, azim) in enumerate([(30, 45), (60, 135), (10, 0)]):
            ax = axes[i]
            ax = fig.add_subplot(1, 3, i+1, projection='3d')
            verts = mesh.vertices
            colors = mesh.visual.vertex_colors[:, :3] / 255.0 if hasattr(mesh.visual, 'vertex_colors') else 'gray'
            ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2],
                           triangles=mesh.faces[:5000],  # 일부만 시각화 (속도)
                           alpha=0.8, edgecolor='none')
            ax.view_init(elev=elev, azim=azim)
            ax.set_title(f'elev={elev}, azim={azim}')
        plt.tight_layout()
        plt.show()
    else:
        print("Mesh file not found.")
else:
    print("No training output found.")

In [ ]:
# 메쉬를 Google Drive에 저장
if train_dirs:
    for name in ["fuse.ply", "fuse_post.ply"]:
        src = os.path.join(train_dirs[-1], name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_SAVE, name))
            print(f"Saved {name} to Google Drive")

## 7. GT Depth 비교 (이 데이터만의 장점)

GT depth가 있으므로 렌더링된 depth와 정량 비교 가능.

In [ ]:
# 렌더링된 depth vs GT depth 비교
# 주의: 렌더링 depth는 정규화된 스케일 (÷100), GT depth는 원본 미터
if test_dirs:
    vis_dir = os.path.join(test_dirs[-1], "vis")
    rendered_depths = sorted(glob.glob(os.path.join(vis_dir, "depth_*.tiff")))

    # test 뷰 인덱스 (eval 모드에서 매 8번째)
    test_indices = list(range(0, len(rgb_files), 8))

    if rendered_depths and test_indices:
        n_compare = min(4, len(rendered_depths), len(test_indices))
        fig, axes = plt.subplots(3, n_compare, figsize=(4 * n_compare, 12))
        if n_compare == 1:
            axes = axes.reshape(3, 1)

        mae_list = []
        for i in range(n_compare):
            # Rendered depth (정규화된 스케일)
            rd = np.array(Image.open(rendered_depths[i]))

            # GT depth → 같은 스케일로 변환
            gt_idx = test_indices[i]
            gt_path = os.path.join(DATA, "depth_gt", depth_files[gt_idx])
            gt_raw = np.array(Image.open(gt_path)).astype(np.float32)
            gt_depth = gt_raw * 0.01 / SCALE_FACTOR  # 정규화된 스케일로 변환

            # 전경 마스크
            fg = gt_raw > 0

            # 오차 계산 (전경 영역만)
            if fg.any() and rd.shape == gt_depth.shape:
                error = np.abs(rd - gt_depth) * fg
                mae = error[fg].mean()
                mae_list.append(mae)
            else:
                mae = float('nan')

            axes[0, i].imshow(gt_depth, cmap='turbo')
            axes[0, i].set_title('GT Depth (normalized)', fontsize=9)
            axes[0, i].axis('off')

            axes[1, i].imshow(rd, cmap='turbo')
            axes[1, i].set_title('Rendered Depth', fontsize=9)
            axes[1, i].axis('off')

            vmax = np.percentile(error[fg], 95) if fg.any() else 1
            axes[2, i].imshow(error, cmap='hot', vmin=0, vmax=vmax)
            axes[2, i].set_title(f'|Error| MAE={mae:.4f}', fontsize=9)
            axes[2, i].axis('off')

        plt.suptitle(f'Depth Comparison (normalized scale) — Mean MAE: {np.nanmean(mae_list):.4f}')
        plt.tight_layout()
        plt.show()
    else:
        print("Cannot compare depths — missing rendered or GT files.")

## 8. (Optional) Ablation — Regularization 효과 비교

텍스처 없는 씬에서 regularization이 얼마나 중요한지 확인.

In [ ]:
# Regularization 없는 baseline (빠른 확인용 7000 iter)
OUTPUT_NOREG = os.path.join(ROOT, "output_no_reg")

# 이 셀은 선택적 — 실행하면 추가 ~15분 소요
# !cd {REPO} && python train.py \
#     -s {DATA} \
#     -m {OUTPUT_NOREG} \
#     --eval \
#     --sh_degree 1 \
#     --lambda_normal 0.0 \
#     --lambda_dist 0.0 \
#     --depth_ratio 1.0 \
#     --iterations 7000 \
#     --test_iterations 7000 \
#     --save_iterations 7000